In [1]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split


In [2]:
data = pd.read_csv("framingham.csv")


In [3]:
data.head()

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


In [4]:
data = data.drop("education",axis = 1)

In [5]:
data.head()
data.isnull().sum()

male                 0
age                  0
currentSmoker        0
cigsPerDay          29
BPMeds              53
prevalentStroke      0
prevalentHyp         0
diabetes             0
totChol             50
sysBP                0
diaBP                0
BMI                 19
heartRate            1
glucose            388
TenYearCHD           0
dtype: int64

In [6]:
required_columns = ["age","BMI","sysBP","currentSmoker","diabetes","BPMeds","male"]
data = data.dropna(subset = required_columns)

In [7]:
data.isnull().sum()

male                 0
age                  0
currentSmoker        0
cigsPerDay          29
BPMeds               0
prevalentStroke      0
prevalentHyp         0
diabetes             0
totChol             48
sysBP                0
diaBP                0
BMI                  0
heartRate            1
glucose            381
TenYearCHD           0
dtype: int64

In [8]:
data=data.drop(["heartRate","glucose"],axis =1)
# data['totChol'] = pd.to_numeric(data['totChol'], errors='coerce')
# data['cigsPerDay'] = pd.to_numeric(data['cigsPerDay'], errors='coerce')

In [9]:
print(data["totChol"].dtypes)


float64


In [10]:
data["totChol"]

0       195.0
1       250.0
2       245.0
3       225.0
4       285.0
        ...  
4234    207.0
4236    210.0
4237    269.0
4238    185.0
4239    196.0
Name: totChol, Length: 4168, dtype: float64

In [11]:

data["cigsPerDay"] = data["cigsPerDay"].mean()
data["totChol"] = data["totChol"].mean()

In [12]:
data["totChol"]

0       236.706553
1       236.706553
2       236.706553
3       236.706553
4       236.706553
           ...    
4234    236.706553
4236    236.706553
4237    236.706553
4238    236.706553
4239    236.706553
Name: totChol, Length: 4168, dtype: float64

In [13]:
data.isnull().sum()

male               0
age                0
currentSmoker      0
cigsPerDay         0
BPMeds             0
prevalentStroke    0
prevalentHyp       0
diabetes           0
totChol            0
sysBP              0
diaBP              0
BMI                0
TenYearCHD         0
dtype: int64

In [14]:
def calculate_framingham_bmi_risk(row):

    age = row["age"]
    bmi = row["BMI"]
    sbp = row["sysBP"]
    smoker = row["currentSmoker"]
    diabetes = row["diabetes"]
    bp_meds = row["BPMeds"]
    male = row["male"]

    # MEN
    if male == 1:

        beta_age = 3.11296
        beta_bmi = 0.79277

        if bp_meds == 1:
            beta_sbp = 1.92672
        else:
            beta_sbp = 1.85508

        beta_smoking = 0.70953
        beta_diabetes = 0.53160

        baseline_survival = 0.88431
        mean_lp = 23.9388

    # WOMEN
    else:

        beta_age = 2.72107
        beta_bmi = 0.51125

        if bp_meds == 1:
            beta_sbp = 2.88267
        else:
            beta_sbp = 2.81291

        beta_smoking = 0.61868
        beta_diabetes = 0.77763

        baseline_survival = 0.94833
        mean_lp = 26.0145

    # Linear predictor
    lp = (
        beta_age * np.log(age)
        + beta_bmi * np.log(bmi)
        + beta_sbp * np.log(sbp)
        + beta_smoking * smoker
        + beta_diabetes * diabetes
    )

    # 10-year risk
    risk = 1 - baseline_survival ** np.exp(lp - mean_lp)

    return risk * 100
data["risk_score"] = data.apply(
    calculate_framingham_bmi_risk,
    axis=1
)

In [15]:
data.head()

,male,age,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,TenYearCHD,risk_score
0,1,39,0,9.03165,0.0,0,0,0,236.706553,106.0,70.0,26.97,0,3.389605
1,0,46,0,9.03165,0.0,0,0,0,236.706553,121.0,81.0,28.73,0,3.529530
2,1,48,1,9.03165,0.0,0,0,0,236.706553,127.5,80.0,25.34,0,16.422024
3,0,61,1,9.03165,0.0,0,1,0,236.706553,150.0,95.0,28.58,1,23.081884
4,0,46,1,9.03165,0.0,0,0,0,236.706553,130.0,84.0,23.10,0,7.041166


In [16]:
X = data.drop("risk_score",axis = 1)
Y = data["risk_score"]

In [17]:
X_train,X_test,Y_train,y_test = train_test_split(
    X,Y , test_size = 0.28, random_state = 42
)

In [18]:
from sklearn.ensemble import RandomForestRegressor
model1 = RandomForestRegressor(n_estimators=100, criterion='squared_error', max_depth=None, min_samples_split=2,
                                min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_features=1.0, max_leaf_nodes=None, min_impurity_decrease=0.0,
                                bootstrap=True, oob_score=False, n_jobs=None,
                                random_state=None, verbose=0, warm_start=False, ccp_alpha=0.0, max_samples=None, monotonic_cst=None)


In [19]:
from sklearn.ensemble import GradientBoostingRegressor
model2 = GradientBoostingRegressor( loss='squared_error', learning_rate=0.1, n_estimators=100, subsample=1.0,
                                   criterion='squared_error', min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_depth=3,
                                   min_impurity_decrease=0.0, init=None, random_state=None, max_features=None, alpha=0.9, verbose=0, max_leaf_nodes=None,
                                   warm_start=False, validation_fraction=0.1, n_iter_no_change=None, tol=0.0001, ccp_alpha=0.0)

In [20]:
data.describe()


,male,age,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,TenYearCHD,risk_score
count,4168.000000,4168.000000,4168.000000,4.168000e+03,4168.000000,4168.000000,4168.000000,4168.000000,4.168000e+03,4168.000000,4168.000000,4168.000000,4168.000000,4168.000000
mean,0.431862,49.521113,0.495441,9.031650e+00,0.029511,0.005278,0.309021,0.025192,2.367066e+02,132.273512,82.901751,25.803486,0.149472,12.405767
std,0.495395,8.542138,0.500039,6.804263e-13,0.169253,0.072469,0.462145,0.156727,7.333681e-12,21.914081,11.872033,4.076441,0.356596,10.760963
min,0.000000,32.000000,0.000000,9.031650e+00,0.000000,0.000000,0.000000,0.000000,2.367066e+02,83.500000,48.000000,15.540000,0.000000,0.785300
25%,0.000000,42.000000,0.000000,9.031650e+00,0.000000,0.000000,0.000000,0.000000,2.367066e+02,117.000000,75.000000,23.070000,0.000000,4.895848
50%,0.000000,49.000000,0.000000,9.031650e+00,0.000000,0.000000,0.000000,0.000000,2.367066e+02,128.000000,82.000000,25.405000,0.000000,9.048025
75%,1.000000,56.000000,1.000000,9.031650e+00,0.000000,0.000000,1.000000,0.000000,2.367066e+02,143.500000,89.625000,28.040000,0.000000,16.514902
max,1.000000,70.000000,1.000000,9.031650e+00,1.000000,1.000000,1.000000,1.000000,2.367066e+02,295.000000,142.500000,56.800000,1.000000,93.612779


In [21]:
model1.fit(X_train , Y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [22]:
print(X.dtypes)

male                 int64
age                  int64
currentSmoker        int64
cigsPerDay         float64
BPMeds             float64
prevalentStroke      int64
prevalentHyp         int64
diabetes             int64
totChol            float64
sysBP              float64
diaBP              float64
BMI                float64
TenYearCHD           int64
dtype: object


In [23]:
model2.fit(X_train , Y_train)




,loss,'squared_error'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'squared_error'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [24]:
y_pred1 = model1.predict(X_test)
y_pred2 = model2.predict(X_test)

In [25]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score



mae = mean_absolute_error(y_test, y_pred1)
mse = mean_squared_error(y_test, y_pred1)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred1)

print(" model 1 MAE:", mae)
print("model 1 RMSE:", rmse)
print("model 1 R²:", r2)



mae = mean_absolute_error(y_test, y_pred2)
mse = mean_squared_error(y_test, y_pred2)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred2)

print(" model 2 MAE:", mae)
print("model 2 RMSE:", rmse)
print("model 2 R²:", r2)


 model 1 MAE: 0.7739214563408248
model 1 RMSE: 1.9805808630528121
model 1 R²: 0.9670058678141084
 model 2 MAE: 1.1441048011416535
model 2 RMSE: 1.722000786116674
model 2 R²: 0.9750587483753764


#  Select Model 1 because of high RMSE value

In [26]:
import joblib 
joblib.dump(model1 ,"heart_model.pkl")

['heart_model.pkl']